# 손실 함수 전략 6종으로 Swiss Roll 언폴딩 비교

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gshong-ai/ADA26/blob/claude/mnist-dimensionality-reduction-H2Meg/04_autoencoder_manifold/autoencoder_swiss_roll_add_loss.ipynb)

## 개요

기준 AE(`autoencoder_swiss_roll.ipynb`)에서 `MLP-16` 구조를 유지하되,
**손실 함수만 달리해서** Swiss Roll 언폴딩 품질이 어떻게 달라지는지 비교합니다.

| # | 방법 | 핵심 아이디어 | 지도 필요 |
|---|------|-------------|---------|
| 1 | **Base AE** | MSE 재구성만 | ❌ |
| 2 | **Contractive AE** | MSE + Jacobian Frobenius 노름 패널티 | ❌ |
| 3 | **Denoising AE** | 노이즈 입력 → 원본 복원 | ❌ |
| 4 | **Neighbor-preserving AE** | MSE + 배치 내 쌍별 거리 보존 패널티 | ❌ |
| 5 | **Multi-task AE** | MSE + t 예측 보조 헤드 (반지도) | ✅ t 필요 |
| 6 | **4th I/O AE** | t를 4번째 입출력으로 추가 (지도) | ✅ t 필요 |

방법 5·6은 매니폴드 파라미터 `t`(정답)를 사용하는 **비교 실험**입니다.
색상(t)이 무지개처럼 연속적일수록 잘 언폴딩된 것입니다.

## 0. 패키지 설치

In [ ]:
!pip install -q plotly scikit-learn tensorflow

## 1. 라이브러리 임포트

In [ ]:
# ─── 라이브러리 임포트 ───
import numpy as np
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots

from sklearn.datasets import make_swiss_roll
from sklearn.manifold import Isomap
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import NearestNeighbors
from scipy.stats import spearmanr

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

pio.renderers.default = 'colab'
print(f'TensorFlow: {tf.__version__}')
print('라이브러리 로드 완료')

## 2. 하이퍼파라미터 & 설정

In [ ]:
# ─── 공통 설정 ───
RANDOM_SEED   = 42
N_SAMPLES     = 2000
SWISS_NOISE   = 0.1

EPOCHS        = 200        # 6개 방법 학습이므로 200으로 조정
BATCH_SIZE    = 256
LEARNING_RATE = 1e-3
LATENT_DIM    = 2
HIDDEN_SIZE   = 16         # 모든 AE 공통 MLP-16 구조

# ─── 방법별 추가 하이퍼파라미터 ───
LAMBDA_C      = 1e-3       # Contractive AE: Jacobian 패널티 강도
NOISE_STD     = 0.5        # Denoising AE: 입력 노이즈 표준편차
MU_NEIGHBOR   = 0.3        # Neighbor AE: 이웃 거리 패널티 강도
ALPHA_T       = 1.0        # Multi-task AE: t 예측 손실 가중치

# ─── Isomap 비교 기준선 ───
ISOMAPNEIGHBORS = 10

# ─── 시각화 ───
OUTPUT_HTML   = 'autoencoder_swiss_roll_add_loss.html'
COLORSCALE    = 'Rainbow'

tf.random.set_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
print('설정 완료')

## 3. 데이터 로드 & 전처리

- `X`: 표준화된 3D Swiss Roll 좌표 (모든 방법 공통)
- `t_norm`: 정규화된 매니폴드 파라미터 (방법 5·6에서 지도 정보로 사용)
- `X4`: t를 4번째 차원으로 붙인 입력 (방법 6 전용)

In [ ]:
# ─── Swiss Roll 생성 ───
X_raw, t = make_swiss_roll(n_samples=N_SAMPLES, noise=SWISS_NOISE,
                            random_state=RANDOM_SEED)

scaler = StandardScaler()
X = scaler.fit_transform(X_raw).astype('float32')

# t 정규화 (Multi-task/4th I/O AE 학습용)
t_norm = ((t - t.mean()) / t.std()).astype('float32')

# 방법 6 전용: (x, y, z, t_norm) 4D 입력
X4 = np.concatenate([X, t_norm[:, None]], axis=1).astype('float32')

# ─── 학습/검증 분할 (90/10) ───
val_size  = int(0.1 * N_SAMPLES)
X_val,  X_train  = X[:val_size],  X[val_size:]
t_val,  t_train  = t_norm[:val_size], t_norm[val_size:]
X4_val, X4_train = X4[:val_size], X4[val_size:]

# ─── TF 데이터셋 ───
def make_ds(x_data, t_data=None):
    """x_data, 또는 (x_data, t_data) 쌍을 배치 데이터셋으로 변환"""
    if t_data is not None:
        ds = tf.data.Dataset.from_tensor_slices((x_data, t_data))
    else:
        ds = tf.data.Dataset.from_tensor_slices(x_data)
    return ds.shuffle(len(x_data), seed=RANDOM_SEED).batch(BATCH_SIZE)

train_ds   = make_ds(X_train)              # 방법 1~4
train_ds_t = make_ds(X_train, t_train)    # 방법 5 (multi-task)
train_ds_4 = make_ds(X4_train)            # 방법 6 (4th I/O)

print(f'Swiss Roll shape: {X.shape}')
print(f'학습 {X_train.shape[0]}개, 검증 {X_val.shape[0]}개')
print(f't 범위: {t.min():.2f} ~ {t.max():.2f}')

## 4. 공통 모델 빌더 & 학습 루프

모든 방법이 동일한 `MLP-16` 구조를 공유합니다:
```
입력(D) → Dense(16, relu) → Dense(2, linear) [bottleneck]
         → Dense(16, relu) → Dense(D, linear) [재구성]
```
방법마다 손실 함수만 달리하여 공정하게 비교합니다.

In [ ]:
# ─── 공통 빌더: 인코더 / 디코더 ───
def build_encoder_decoder(input_dim=3, hidden_size=HIDDEN_SIZE,
                          latent_dim=LATENT_DIM):
    """Keras Functional API로 인코더/디코더 쌍 생성"""
    # 인코더
    inp  = keras.Input(shape=(input_dim,), name='input')
    h_e  = layers.Dense(hidden_size, activation='relu',   name='enc_hidden')(inp)
    z    = layers.Dense(latent_dim,  activation='linear', name='bottleneck')(h_e)
    encoder = keras.Model(inp, z, name=f'encoder_{input_dim}d')

    # 디코더
    z_in = keras.Input(shape=(latent_dim,), name='z_input')
    h_d  = layers.Dense(hidden_size, activation='relu',   name='dec_hidden')(z_in)
    out  = layers.Dense(input_dim,  activation='linear', name='output')(h_d)
    decoder = keras.Model(z_in, out, name=f'decoder_{input_dim}d')

    return encoder, decoder


def build_t_pred_head(latent_dim=LATENT_DIM):
    """t 예측 보조 헤드 (방법 5 전용)"""
    z_in = keras.Input(shape=(latent_dim,), name='z_for_t')
    h    = layers.Dense(8, activation='relu',   name='t_hidden')(z_in)
    out  = layers.Dense(1, activation='linear', name='t_out')(h)
    return keras.Model(z_in, out, name='t_pred_head')


# ─── 공통 학습 루프 ───
def train_loop(step_fn, val_fn, dataset, epochs, label):
    """
    step_fn(batch) -> (total_loss, ...)  첫 번째 원소가 total loss
    val_fn()       -> val_loss
    """
    history = {'loss': [], 'val_loss': []}
    for epoch in range(epochs):
        batch_losses = [float(step_fn(batch)[0]) for batch in dataset]
        val_loss = float(val_fn())
        history['loss'].append(np.mean(batch_losses))
        history['val_loss'].append(val_loss)
        if (epoch + 1) % 100 == 0:
            print(f'  [{label}] Epoch {epoch+1:3d}/{epochs}'
                  f' — train: {np.mean(batch_losses):.5f}, val: {val_loss:.5f}')
    return history

print('모델 빌더 & 학습 루프 준비 완료')

## 5. 방법 1 — Base AE (기준)

**손실:** MSE 재구성 오차만 사용
$$\mathcal{L} = \frac{1}{N}\|\mathbf{x} - \hat{\mathbf{x}}\|^2$$

In [ ]:
print('=== 방법 1: Base AE (기준 MSE) ===')
enc1, dec1 = build_encoder_decoder()
opt1  = keras.optimizers.Adam(LEARNING_RATE)
vars1 = enc1.trainable_variables + dec1.trainable_variables

@tf.function
def step1(x_batch):
    with tf.GradientTape() as tape:
        z    = enc1(x_batch, training=True)
        x_r  = dec1(z, training=True)
        loss = tf.reduce_mean(tf.square(x_batch - x_r))
    opt1.apply_gradients(zip(tape.gradient(loss, vars1), vars1))
    return (loss,)

def val1():
    z = enc1(X_val, training=False)
    return tf.reduce_mean(tf.square(X_val - dec1(z, training=False)))

hist1 = train_loop(step1, val1, train_ds, EPOCHS, label='Base AE')
print('완료')

## 6. 방법 2 — Contractive AE (CAE)

인코더 출력 $\mathbf{z}$의 입력 $\mathbf{x}$에 대한 **Jacobian Frobenius 노름**을 패널티로 추가합니다.
입력이 조금 변해도 잠재 표현이 안정적이도록 강제 → 매니폴드에 민감한 방향만 인코딩.

$$\mathcal{L} = \text{MSE} + \lambda \cdot \left\|\frac{\partial\mathbf{z}}{\partial\mathbf{x}}\right\|_F^2$$

`tf.GradientTape.batch_jacobian`으로 배치별 야코비안을 계산합니다.

In [ ]:
print('=== 방법 2: Contractive AE (Jacobian 패널티) ===')
enc2, dec2 = build_encoder_decoder()
opt2  = keras.optimizers.Adam(LEARNING_RATE)
vars2 = enc2.trainable_variables + dec2.trainable_variables

@tf.function
def step2(x_batch):
    with tf.GradientTape(persistent=True) as tape:
        tape.watch(x_batch)          # 입력에 대한 미분을 위해 감시
        z    = enc2(x_batch, training=True)
        x_r  = dec2(z, training=True)
        recon = tf.reduce_mean(tf.square(x_batch - x_r))
        # Jacobian ∂z/∂x: shape (batch, latent_dim, input_dim)
        jac   = tape.batch_jacobian(z, x_batch)
        # Frobenius 노름 제곱 평균
        contr = LAMBDA_C * tf.reduce_mean(
            tf.reduce_sum(tf.square(jac), axis=[1, 2])
        )
        loss = recon + contr
    grads = tape.gradient(loss, vars2)
    del tape  # persistent tape 명시적 해제
    opt2.apply_gradients(zip(grads, vars2))
    return (loss,)

def val2():
    z = enc2(X_val, training=False)
    return tf.reduce_mean(tf.square(X_val - dec2(z, training=False)))

hist2 = train_loop(step2, val2, train_ds, EPOCHS, label='Contractive AE')
print('완료')

## 7. 방법 3 — Denoising AE (DAE)

**손실:** 가우시안 노이즈가 추가된 입력 → 원본(clean) 재구성

$$\mathcal{L} = \frac{1}{N}\|\mathbf{x} - f(\mathbf{x} + \epsilon)\|^2, \quad \epsilon \sim \mathcal{N}(0, \sigma^2)$$

데이터 포인트는 매니폴드 위에 있지만 노이즈는 매니폴드 바깥으로 이탈합니다.
원본을 복원하려면 모델이 매니폴드 방향을 파악해야 합니다.

In [ ]:
print('=== 방법 3: Denoising AE (노이즈 제거) ===')
enc3, dec3 = build_encoder_decoder()
opt3  = keras.optimizers.Adam(LEARNING_RATE)
vars3 = enc3.trainable_variables + dec3.trainable_variables

@tf.function
def step3(x_batch):
    # 학습 시: 노이즈 추가 → 원본 복원
    x_noisy = x_batch + tf.random.normal(tf.shape(x_batch), stddev=NOISE_STD)
    with tf.GradientTape() as tape:
        z    = enc3(x_noisy, training=True)
        x_r  = dec3(z, training=True)
        loss = tf.reduce_mean(tf.square(x_batch - x_r))  # 원본과 비교
    opt3.apply_gradients(zip(tape.gradient(loss, vars3), vars3))
    return (loss,)

def val3():
    # 검증 시: 노이즈 없이 평가
    z = enc3(X_val, training=False)
    return tf.reduce_mean(tf.square(X_val - dec3(z, training=False)))

hist3 = train_loop(step3, val3, train_ds, EPOCHS, label='Denoising AE')
print('완료')

## 8. 방법 4 — Neighbor-preserving AE

배치 내 쌍별 거리를 3D 입력 공간과 2D 잠재 공간에서 각각 계산하고,
그 차이를 페널티로 추가합니다.

$$\mathcal{L} = \text{MSE} + \mu \cdot \frac{1}{B^2}\sum_{i,j}\left(\tilde{d}_z^{ij} - \tilde{d}_x^{ij}\right)^2$$

$\tilde{d}$: 배치 내 최대값으로 정규화한 거리 (스케일 독립적 비교)

In [ ]:
print('=== 방법 4: Neighbor-preserving AE (이웃 거리 보존) ===')
enc4, dec4 = build_encoder_decoder()
opt4  = keras.optimizers.Adam(LEARNING_RATE)
vars4 = enc4.trainable_variables + dec4.trainable_variables

@tf.function
def pairwise_dist(a):
    """배치 내 쌍별 유클리드 거리 행렬 (B × B)"""
    r    = tf.reduce_sum(a * a, axis=1, keepdims=True)
    d_sq = r - 2.0 * tf.matmul(a, tf.transpose(a)) + tf.transpose(r)
    return tf.sqrt(tf.maximum(d_sq, 1e-8))

@tf.function
def step4(x_batch):
    with tf.GradientTape() as tape:
        z    = enc4(x_batch, training=True)
        x_r  = dec4(z, training=True)
        recon = tf.reduce_mean(tf.square(x_batch - x_r))
        # 쌍별 거리 행렬
        d_x = pairwise_dist(x_batch)
        d_z = pairwise_dist(z)
        # 최대값으로 정규화 (스케일 독립적)
        d_x_n = d_x / (tf.reduce_max(d_x) + 1e-8)
        d_z_n = d_z / (tf.reduce_max(d_z) + 1e-8)
        # 이웃 거리 보존 패널티
        neighbor = MU_NEIGHBOR * tf.reduce_mean(tf.square(d_z_n - d_x_n))
        loss = recon + neighbor
    opt4.apply_gradients(zip(tape.gradient(loss, vars4), vars4))
    return (loss,)

def val4():
    z = enc4(X_val, training=False)
    return tf.reduce_mean(tf.square(X_val - dec4(z, training=False)))

hist4 = train_loop(step4, val4, train_ds, EPOCHS, label='Neighbor AE')
print('완료')

## 9. 방법 5 — Multi-task AE (t 예측 보조 헤드)

**반지도 학습:** 입력은 3D 그대로 유지하되, bottleneck에서 `t`를 예측하는
보조 헤드를 추가합니다.

```
입력(3D) → 인코더 → bottleneck(2D) ─┬─ 디코더 → 재구성(3D)  [주 목표]
                                     └─ t-헤드 → t 예측(1D)  [보조 목표]
```

$$\mathcal{L} = \text{MSE}_{\text{recon}} + \alpha \cdot \text{MSE}_t$$

> ⚠️ `t`가 외부에서 제공되므로 순수 비지도 학습이 아닙니다.

In [ ]:
print('=== 방법 5: Multi-task AE (t 예측 보조 헤드) ===')
enc5, dec5 = build_encoder_decoder()
t_head5   = build_t_pred_head()
opt5      = keras.optimizers.Adam(LEARNING_RATE)
vars5     = (enc5.trainable_variables
             + dec5.trainable_variables
             + t_head5.trainable_variables)

@tf.function
def step5(batch):
    x_batch, t_batch = batch
    with tf.GradientTape() as tape:
        z      = enc5(x_batch, training=True)
        x_r    = dec5(z, training=True)
        t_pred = t_head5(z, training=True)
        # 재구성 손실
        recon  = tf.reduce_mean(tf.square(x_batch - x_r))
        # t 예측 보조 손실
        t_loss = ALPHA_T * tf.reduce_mean(
            tf.square(t_batch - tf.squeeze(t_pred, axis=-1))
        )
        loss = recon + t_loss
    opt5.apply_gradients(zip(tape.gradient(loss, vars5), vars5))
    return (loss,)

def val5():
    z = enc5(X_val, training=False)
    return tf.reduce_mean(tf.square(X_val - dec5(z, training=False)))

hist5 = train_loop(step5, val5, train_ds_t, EPOCHS, label='Multi-task AE')
print('완료')

## 10. 방법 6 — 4th I/O AE (t를 4번째 입출력으로 추가)

**지도 학습:** 입력과 출력 모두 `(x, y, z, t)` 4D로 확장합니다.

```
입력(x, y, z, t_norm) → 인코더 → bottleneck(2D) → 디코더 → 재구성(x, y, z, t_norm)
```

$$\mathcal{L} = \text{MSE}_{4D}$$

> ⚠️ bottleneck이 `z₁ ≈ t, z₂ ≈ height`로 수렴할 가능성이 높습니다.
> 발견(unsupervised)이 아닌 암기(supervised)에 해당합니다.

In [ ]:
print('=== 방법 6: 4th I/O AE (t 입출력 포함, 지도) ===')
enc6, dec6 = build_encoder_decoder(input_dim=4)  # 4D 입출력
opt6  = keras.optimizers.Adam(LEARNING_RATE)
vars6 = enc6.trainable_variables + dec6.trainable_variables

@tf.function
def step6(x4_batch):
    with tf.GradientTape() as tape:
        z    = enc6(x4_batch, training=True)
        x4_r = dec6(z, training=True)
        loss = tf.reduce_mean(tf.square(x4_batch - x4_r))
    opt6.apply_gradients(zip(tape.gradient(loss, vars6), vars6))
    return (loss,)

def val6():
    z = enc6(X4_val, training=False)
    return tf.reduce_mean(tf.square(X4_val - dec6(z, training=False)))

hist6 = train_loop(step6, val6, train_ds_4, EPOCHS, label='4th I/O AE')
print('완료')

## 11. 비교 기준선: Isomap

측지 거리 기반의 비선형 비지도 방법. 비교 상한선 역할을 합니다.

In [ ]:
print(f'Isomap 투영 중... (n_neighbors={ISOMAPNEIGHBORS})')
isomap = Isomap(n_components=2, n_neighbors=ISOMAPNEIGHBORS)
coords_iso = isomap.fit_transform(X)
print(f'Isomap 재구성 오차: {isomap.reconstruction_error():.4f}')

## 12. 2D 좌표 추출

In [ ]:
# ─── 각 방법의 2D bottleneck 좌표 추출 ───
coords1 = enc1.predict(X,  verbose=0)   # Base AE
coords2 = enc2.predict(X,  verbose=0)   # Contractive AE
coords3 = enc3.predict(X,  verbose=0)   # Denoising AE
coords4 = enc4.predict(X,  verbose=0)   # Neighbor AE
coords5 = enc5.predict(X,  verbose=0)   # Multi-task AE (입력은 3D)
coords6 = enc6.predict(X4, verbose=0)   # 4th I/O AE (입력은 4D)
coords7 = coords_iso                    # Isomap

print('2D 좌표 추출 완료')
for name, c in [('Base', coords1), ('CAE', coords2), ('DAE', coords3),
                ('Neighbor', coords4), ('Multi-task', coords5),
                ('4th I/O', coords6), ('Isomap', coords7)]:
    print(f'  {name:12s}: shape={c.shape}, '
          f'z₁=[{c[:,0].min():.2f},{c[:,0].max():.2f}], '
          f'z₂=[{c[:,1].min():.2f},{c[:,1].max():.2f}]')

## 13. 평가 & 시각화 — 2D 투영 비교

색상(t)이 **무지개처럼 연속적**일수록 Swiss Roll이 잘 언폴딩된 것입니다.

In [ ]:
# ─── 2D 투영 비교 시각화 (2 × 4 그리드, 7개 사용) ───
plot_items = [
    ('Base AE\n(기준 MSE)',          coords1),
    ('Contractive AE\n(Jacobian)',   coords2),
    ('Denoising AE\n(노이즈 제거)',   coords3),
    ('Neighbor AE\n(이웃 보존)',       coords4),
    ('Multi-task AE\n(t 예측 보조)', coords5),
    ('4th I/O AE\n(t 입출력, 지도)', coords6),
    ('Isomap\n(비선형 기준선)',        coords7),
]

fig = make_subplots(
    rows=2, cols=4,
    subplot_titles=[label.replace('\n', ' ') for label, _ in plot_items] + [''],
    horizontal_spacing=0.06,
    vertical_spacing=0.14,
)

positions = [(1,1),(1,2),(1,3),(1,4),(2,1),(2,2),(2,3)]

for (label, coords), (row, col) in zip(plot_items, positions):
    show_bar = (row == 1 and col == 4)
    fig.add_trace(
        go.Scatter(
            x=coords[:, 0], y=coords[:, 1],
            mode='markers',
            marker=dict(
                size=3, color=t, colorscale=COLORSCALE, opacity=0.8,
                showscale=show_bar,
                colorbar=dict(title='t', x=1.02) if show_bar else None,
            ),
            showlegend=False,
        ),
        row=row, col=col,
    )

fig.update_layout(
    title=dict(
        text=('Swiss Roll 2D 투영 비교: 6가지 손실 전략 + Isomap<br>'
              '<sup>색상(t)이 무지개처럼 연속적일수록 매니폴드가 잘 펼쳐진 것'
              '  |  방법 5·6: t 사용 (지도/반지도)</sup>'),
        x=0.5, font=dict(size=15),
    ),
    width=1200, height=680,
    paper_bgcolor='white',
)
fig.show()
fig.write_html(OUTPUT_HTML, include_plotlyjs='cdn')
print(f'저장 완료: {OUTPUT_HTML}')

## 14. 학습 곡선 비교

손실 함수가 달라서 절댓값 비교는 무의미합니다.
**추세(수렴 속도, 안정성)**에 집중해서 봅니다.

In [ ]:
# ─── 학습 곡선 비교 ───
method_info = [
    ('Base AE',        hist1, '#95a5a6'),
    ('Contractive AE', hist2, '#3498DB'),
    ('Denoising AE',   hist3, '#2ECC71'),
    ('Neighbor AE',    hist4, '#E74C3C'),
    ('Multi-task AE',  hist5, '#9B59B6'),
    ('4th I/O AE',     hist6, '#F39C12'),
]

fig_loss = go.Figure()
for name, hist, color in method_info:
    fig_loss.add_trace(go.Scatter(
        y=hist['val_loss'],
        mode='lines',
        name=name,
        line=dict(color=color, width=2),
    ))

fig_loss.update_layout(
    title='검증 손실(MSE) 학습 곡선 — 6가지 방법 비교<br>'
          '<sup>손실 정의가 달라 절댓값 비교는 무의미, 추세·안정성 위주로 확인</sup>',
    xaxis_title='Epoch',
    yaxis_title='Validation MSE (재구성 오차)',
    yaxis_type='log',
    width=900, height=450,
    legend=dict(x=0.75, y=0.95),
)
fig_loss.show()

print('=== 최종 검증 MSE 요약 ===')
for name, hist, _ in method_info:
    print(f'  {name:18s}: {hist["val_loss"][-1]:.6f}')

## 15. 매니폴드 연속성 정량 평가

2D 좌표에서 k-최근접 이웃의 **매니폴드 파라미터 t 분산 평균**을 계산합니다.
- **낮을수록** → 2D에서 가까운 점들이 매니폴드에서도 가까움 → 잘 펼쳐진 것
- `t`를 평가 지표로 사용하므로, 방법 5·6의 점수는 `t`를 학습에 사용했기 때문에 높게 나올 수 있습니다 (순환 논리 주의)

In [ ]:
# ─── 연속성 평가 함수 ───
def continuity_score(coords_2d, t_vals, k=10):
    """k-이웃 t값 분산 평균 (낮을수록 잘 펼쳐진 것)"""
    nbrs = NearestNeighbors(n_neighbors=k + 1).fit(coords_2d)
    _, indices = nbrs.kneighbors(coords_2d)
    scores = [np.var(t_vals[indices[i, 1:]]) for i in range(len(coords_2d))]
    return np.mean(scores)

# ─── 전체 방법 평가 ───
eval_items = [
    ('Base AE',        coords1, '#95a5a6', '비지도'),
    ('Contractive AE', coords2, '#3498DB', '비지도'),
    ('Denoising AE',   coords3, '#2ECC71', '비지도'),
    ('Neighbor AE',    coords4, '#E74C3C', '비지도'),
    ('Multi-task AE',  coords5, '#9B59B6', '반지도 ⚠️'),
    ('4th I/O AE',     coords6, '#F39C12', '지도 ⚠️'),
    ('Isomap',         coords7, '#1ABC9C', '비지도'),
]

print('=== 매니폴드 연속성 평가 ===')
print('(이웃 t값 분산 평균: 낮을수록 잘 펼쳐진 것)')
print()

scores = []
for name, coords, _, kind in eval_items:
    s = continuity_score(coords, t)
    scores.append(s)
    print(f'  {name:18s} [{kind:10s}]: {s:.4f}')

# 막대 차트
fig_cont = go.Figure(go.Bar(
    x=[name for name, *_ in eval_items],
    y=scores,
    marker_color=[color for _, _, color, _ in eval_items],
    opacity=0.85,
    text=[f'{s:.4f}' for s in scores],
    textposition='auto',
))
fig_cont.update_layout(
    title='매니폴드 연속성 평가 — 이웃 t값 평균 분산 (↓ 낮을수록 잘 펼쳐진 것)<br>'
          '<sup>⚠️ 방법 5·6은 t를 학습에 사용하므로 순환 논리 주의</sup>',
    xaxis_title='방법',
    yaxis_title='이웃 t값 평균 분산',
    width=900, height=480,
    showlegend=False,
)
fig_cont.show()

## 16. 결론

### 비지도 방법 (t 불필요) 비교

| 방법 | 핵심 아이디어 | 기대 효과 | 한계 |
|------|-------------|----------|------|
| Base AE | MSE만 | 기준선 | 위상 구조 무시 |
| Contractive AE | Jacobian 압축 | 매니폴드 접선 방향만 민감 | λ 튜닝 필요 |
| Denoising AE | 노이즈 제거 | 매니폴드 투영 학습 | 노이즈 크기 민감 |
| Neighbor AE | 거리 보존 | 위상 구조 보존 | 배치 크기 의존 |

### 지도/반지도 방법 (t 필요) 비교

| 방법 | 성격 | 문제점 |
|------|------|-------|
| Multi-task AE | 반지도 | 입력은 순수하지만 t로 bottleneck 유도 |
| 4th I/O AE | 지도 | t가 bottleneck에 직접 압축됨 — 정답 암기 |

### 핵심 메시지

> **손실 함수는 모델이 "무엇에 집중할지"를 결정합니다.**
>
> - `t`를 직접 사용하면 언폴딩은 쉽지만, 실제 데이터에서는 `t`가 존재하지 않습니다.
> - 순수 비지도 방법 중에서는 **Contractive AE**와 **Neighbor AE**가
>   매니폴드 구조에 더 직접적인 유도 신호를 제공합니다.
> - 그럼에도 Isomap처럼 "구조를 명시적으로 아는" 기법에는 여전히 미치지 못합니다.

### 더 나아가기
- **Topological Autoencoder**: 지속적 호몰로지(persistent homology) 기반 위상 보존 손실
- **VAE**: 잠재 공간에 정규분포 사전 분포 추가
- **더 깊은 인코더**: 층을 더 쌓아 비선형 표현력 증가